# Virgio Decision Intelligence Pipeline
## Consolidated End-to-End Pipeline — Schedulable via Databricks Jobs

**Architecture:** Raw tables → Clean tables → ML Models → Decision Intelligence → Gold Tables → Genie Space

| Section | Output | Approx Time |
|---|---|---|
| 1. Data Cleaning | `clean_orders`, `clean_daily_metrics`, `clean_sessions`, `clean_product_events` | ~18 min |
| 2. Model v2 (6 improvements) | Trained ensemble + conformal intervals | ~3 min |
| 3. Decision Intelligence | Anomalies, scenarios, causal, recommendations | ~2 min |
| 4. Gold Table Writes | 7 `gold_*` tables | ~1 min |
| 5. Validation | System health check | <10s |

**Model version:** `v3.0_consolidated_pipeline`
**Target:** `ameyam_statsig_databricks_warehouse.default`
**Genie Space:** Virgio Analytics Hub (auto-refreshes from gold tables)

In [0]:
import subprocess, sys

packages = [
    "xgboost", "prophet", "mlflow", "scikit-learn",
    "scipy", "statsmodels", "mapie",
    "dowhy", "econml", "ortools",
    "pandas", "numpy"
]
subprocess.check_call(
    [sys.executable, "-m", "pip", "install"] + packages + ["--quiet"]
)
print("✅ All packages installed")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
googleapis-common-protos 1.65.0 requires protobuf!=3.20.0,!=3.20.1,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0.dev0,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
grpcio-status 1.67.0 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.


✅ All packages installed


In [0]:
import numpy as np
import pandas as pd
import uuid, json
from datetime import datetime, timedelta
from typing import Dict, List, Tuple, Optional

# ML
import xgboost as xgb
from sklearn.linear_model import Ridge
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from scipy import stats
from scipy.stats import ks_2samp
from prophet import Prophet
from mapie.regression import CrossConformalRegressor as MapieRegressor

# Fix circular import (stale module cache after mid-session pip install)
import sys
for _mod in [k for k in sys.modules if any(x in k for x in ['yaml', 'mlflow'])]:
    del sys.modules[_mod]
import yaml
import mlflow

# Spark
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

# Suppress verbose logs
import logging
for logger_name in ['prophet', 'cmdstanpy', 'dowhy']:
    logging.getLogger(logger_name).setLevel(logging.WARNING)

# ======================== CONFIGURATION ========================
CATALOG = "ameyam_statsig_databricks_warehouse"
SCHEMA = "default"
MODEL_VERSION = "v3.0_consolidated_pipeline"
ROLLING_WINDOW_DAYS = 180
TEST_DAYS = 30
TOP_REGIONS = ["Maharashtra", "Karnataka", "Delhi", "Haryana", "Uttar Pradesh"]
RANDOM_STATE = 42

# MLflow
mlflow.set_experiment("/Users/sukriti.shukla@virgio.com/Revenue_Forecasting_Experiment")

print(f"✅ Pipeline initialized: {MODEL_VERSION}")
print(f"   Target: {CATALOG}.{SCHEMA}")
print(f"   Timestamp: {datetime.now()}")

✅ Pipeline initialized: v3.0_consolidated_pipeline
   Target: ameyam_statsig_databricks_warehouse.default
   Timestamp: 2026-06-17 17:56:28.006997


If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


## Section 1: Data Cleaning & Standardization
Builds ALL 4 clean tables from raw source tables:
- `clean_orders` — order lifecycle + financials + attribution (~2 min)
- `clean_daily_metrics` — pre-aggregated KPIs (~30s)
- `clean_sessions` — session data with identity resolution (~3 min)
- `clean_product_events` — filtered browsing events, 16 key types (~10-15 min)

In [0]:
# =============================================================================
# BUILD clean_orders: order_events + shopify financials + session attribution
# OPTIMIZED: Native Spark expressions (no Python UDFs) + no diagnostic counts
# =============================================================================
from itertools import chain

# --- City normalization via native Spark create_map (runs in JVM, 100x faster than UDFs) ---
CITY_MAPPINGS = {
    'bangalore': 'Bengaluru', 'bengaluru': 'Bengaluru', 'banglore': 'Bengaluru',
    'mumbai': 'Mumbai', 'bombay': 'Mumbai', 'navi mumbai': 'Navi Mumbai',
    'new delhi': 'New Delhi', 'delhi': 'New Delhi', 'noida': 'Noida',
    'gurgaon': 'Gurugram', 'gurugram': 'Gurugram',
    'hyderabad': 'Hyderabad', 'chennai': 'Chennai', 'kolkata': 'Kolkata',
    'calcutta': 'Kolkata', 'pune': 'Pune', 'ahmedabad': 'Ahmedabad',
    'jaipur': 'Jaipur', 'lucknow': 'Lucknow', 'chandigarh': 'Chandigarh',
    'kochi': 'Kochi', 'cochin': 'Kochi', 'thiruvananthapuram': 'Thiruvananthapuram',
    'trivandrum': 'Thiruvananthapuram', 'coimbatore': 'Coimbatore',
    'indore': 'Indore', 'bhopal': 'Bhopal', 'nagpur': 'Nagpur',
    'vadodara': 'Vadodara', 'baroda': 'Vadodara', 'surat': 'Surat',
    'visakhapatnam': 'Visakhapatnam', 'vizag': 'Visakhapatnam',
    'patna': 'Patna', 'ranchi': 'Ranchi', 'bhubaneswar': 'Bhubaneswar',
    'guwahati': 'Guwahati', 'dehradun': 'Dehradun', 'mysore': 'Mysuru',
    'mysuru': 'Mysuru', 'mangalore': 'Mangaluru', 'mangaluru': 'Mangaluru',
    'faridabad': 'Faridabad', 'ghaziabad': 'Ghaziabad',
    'greater noida': 'Greater Noida', 'thane': 'Thane'
}

STATE_MAPPINGS = {
    'karnataka': 'Karnataka', 'maharashtra': 'Maharashtra',
    'delhi': 'Delhi', 'new delhi': 'Delhi', 'ncr': 'Delhi',
    'tamil nadu': 'Tamil Nadu', 'tamilnadu': 'Tamil Nadu',
    'telangana': 'Telangana', 'telengana': 'Telangana',
    'uttar pradesh': 'Uttar Pradesh', 'up': 'Uttar Pradesh',
    'west bengal': 'West Bengal', 'westbengal': 'West Bengal',
    'andhra pradesh': 'Andhra Pradesh', 'ap': 'Andhra Pradesh',
    'madhya pradesh': 'Madhya Pradesh', 'mp': 'Madhya Pradesh',
    'rajasthan': 'Rajasthan', 'gujarat': 'Gujarat',
    'haryana': 'Haryana', 'punjab': 'Punjab', 'kerala': 'Kerala',
    'bihar': 'Bihar', 'odisha': 'Odisha', 'orissa': 'Odisha',
    'jharkhand': 'Jharkhand', 'uttarakhand': 'Uttarakhand',
    'uttaranchal': 'Uttarakhand', 'chhattisgarh': 'Chhattisgarh',
    'chattisgarh': 'Chhattisgarh', 'goa': 'Goa', 'assam': 'Assam',
    'himachal pradesh': 'Himachal Pradesh', 'j&k': 'Jammu and Kashmir',
    'jammu and kashmir': 'Jammu and Kashmir', 'jammu & kashmir': 'Jammu and Kashmir',
    'chandigarh': 'Chandigarh', 'puducherry': 'Puducherry',
    'pondicherry': 'Puducherry', 'tripura': 'Tripura',
    'meghalaya': 'Meghalaya', 'manipur': 'Manipur',
    'nagaland': 'Nagaland', 'mizoram': 'Mizoram',
    'arunachal pradesh': 'Arunachal Pradesh', 'sikkim': 'Sikkim'
}

# Build metro lookup: city (lowercase) -> metro name
METRO_CITY_MAP = {}
for metro, cities in {
    'Mumbai Metro': ['Mumbai', 'Navi Mumbai', 'Thane'],
    'NCR': ['New Delhi', 'Noida', 'Gurugram', 'Faridabad', 'Ghaziabad', 'Greater Noida'],
    'Bangalore Metro': ['Bengaluru'],
    'Hyderabad Metro': ['Hyderabad'],
    'Chennai Metro': ['Chennai'],
    'Kolkata Metro': ['Kolkata'],
    'Pune Metro': ['Pune'],
    'Ahmedabad Metro': ['Ahmedabad']
}.items():
    for city in cities:
        METRO_CITY_MAP[city.lower()] = metro

# Create Spark map expressions (JVM-native, no serialization)
city_map_expr = F.create_map([F.lit(x) for x in chain(*CITY_MAPPINGS.items())])
state_map_expr = F.create_map([F.lit(x) for x in chain(*STATE_MAPPINGS.items())])
metro_map_expr = F.create_map([F.lit(x) for x in chain(*METRO_CITY_MAP.items())])

# --- Load source tables (NO .count() calls — saves ~40s) ---
print("Loading source tables...")
df_order_events = spark.table(f"{CATALOG}.{SCHEMA}.order_events")
df_shopify = spark.table(f"{CATALOG}.{SCHEMA}.shopify_orders_raw")
df_osm = spark.table(f"{CATALOG}.{SCHEMA}.order_session_map_v5")
df_identity = spark.table(f"{CATALOG}.{SCHEMA}.identity_bridge_v5")

# --- Join shopify financials (cast STRING → DOUBLE) ---
df_shopify_clean = df_shopify.select(
    F.col("order_name").alias("shopify_order_name"),
    F.col("total_price").cast("double").alias("shopify_total_price"),
    F.col("subtotal_price").cast("double").alias("shopify_subtotal_price"),
    F.col("total_refunded").cast("double").alias("shopify_total_refunded"),
    F.col("total_shipping_price").cast("double").alias("shopify_shipping_price"),
    F.col("net_price").cast("double").alias("shopify_net_price")
)

# --- Join session attribution ---
df_osm_clean = df_osm.select(
    F.col("order_id").alias("osm_order_id"),
    F.col("app_vs_web").alias("osm_app_vs_web"),
    F.col("lt_stable_id"), F.col("lt_session_date"),
    F.col("lt_utm_source").alias("osm_utm_source"),
    F.col("lt_utm_campaign").alias("osm_utm_campaign"),
    F.col("lt_utm_medium").alias("osm_utm_medium")
)

# --- Join identity bridge (deduplicate to prevent fan-out) ---
df_identity_clean = df_identity.select(
    F.col("stable_id").alias("ib_stable_id"),
    F.col("email").alias("resolved_email")
).dropDuplicates(["ib_stable_id"])

# --- Build clean_orders (3 left joins) ---
print("Joining tables...")
df_clean = df_order_events \
    .join(df_shopify_clean, df_order_events["order_name"] == df_shopify_clean["shopify_order_name"], "left") \
    .join(df_osm_clean, df_order_events["order_id"] == df_osm_clean["osm_order_id"], "left") \
    .join(df_identity_clean, df_order_events["stable_id"] == df_identity_clean["ib_stable_id"], "left") \
    .drop("shopify_order_name", "osm_order_id", "ib_stable_id")

# --- Add derived columns (ALL NATIVE SPARK — no Python UDFs) ---
print("Applying normalization (native Spark, no UDFs)...")
city_lower = F.lower(F.trim(F.regexp_replace(F.col("shipping_city"), r'[^a-zA-Z\s]', '')))
city_lower_clean = F.regexp_replace(city_lower, r'\s+', ' ')
state_lower = F.lower(F.trim(F.col("shipping_province")))

df_clean = df_clean \
    .withColumn("_city_key", city_lower_clean) \
    .withColumn("shipping_city_clean",
        F.coalesce(city_map_expr[F.col("_city_key")], F.initcap(F.trim(F.col("shipping_city"))))) \
    .withColumn("shipping_state_clean",
        F.coalesce(state_map_expr[state_lower], F.initcap(F.trim(F.col("shipping_province"))))) \
    .withColumn("shipping_metro_region",
        F.coalesce(metro_map_expr[F.lower(F.col("shipping_city_clean"))], F.lit("Tier 2/3"))) \
    .withColumn("is_identified", F.col("resolved_email").isNotNull()) \
    .withColumn("is_discounted", F.col("total_discounts") > 0) \
    .withColumn("discount_pct",
        F.when(F.col("amount") > 0, F.col("total_discounts") / F.col("amount") * 100).otherwise(0)) \
    .withColumn("order_date", F.to_date("order_created_at")) \
    .withColumn("channel",
        F.when(F.col("osm_app_vs_web") == "App", "App")
        .when(F.col("osm_app_vs_web") == "Web", "Web")
        .otherwise(F.coalesce(F.col("app_vs_web"), F.lit("Unknown")))) \
    .drop("_city_key")

# --- Write clean_orders ---
print("Writing to Delta...")
df_clean.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{SCHEMA}.clean_orders")

co_count = spark.table(f"{CATALOG}.{SCHEMA}.clean_orders").count()
print(f"\n✅ clean_orders written: {co_count:,} rows")

Loading source tables...
Joining tables...
Applying normalization (native Spark, no UDFs)...
Writing to Delta...

✅ clean_orders written: 456,016 rows


In [0]:
# =============================================================================
# BUILD clean_daily_metrics + LOAD clean_sessions
# =============================================================================

df_co = spark.table(f"{CATALOG}.{SCHEMA}.clean_orders")
df_placed = df_co.filter(F.col("event_name") == "Order Placed")

# Aggregate daily metrics
df_daily = df_placed.groupBy(
    "order_date", "shipping_state_clean", "shipping_city_clean",
    "shipping_metro_region", "channel"
).agg(
    F.count("*").alias("orders_placed"),
    F.sum("amount").alias("gross_revenue"),
    F.sum("total_discounts").alias("total_discounts"),
    F.avg("amount").alias("avg_order_value"),
    F.sum(F.when(F.col("is_discounted") == True, 1).otherwise(0)).alias("discounted_orders"),
    F.avg("discount_pct").alias("avg_discount_pct"),
    F.countDistinct("customer_id").alias("unique_customers"),
    F.sum(F.when(F.col("event_name") == "Item Returned", 1).otherwise(0)).alias("items_returned"),
    F.sum(F.when(F.col("event_name") == "Item Cancelled", 1).otherwise(0)).alias("items_cancelled")
)

# Add calculated metrics
df_daily = df_daily \
    .withColumn("net_revenue", F.col("gross_revenue") - F.col("total_discounts")) \
    .withColumn("discount_penetration_pct",
        F.when(F.col("orders_placed") > 0, F.col("discounted_orders") / F.col("orders_placed") * 100).otherwise(0)) \
    .withColumn("return_rate_pct",
        F.when(F.col("orders_placed") > 0, F.col("items_returned") / F.col("orders_placed") * 100).otherwise(0)) \
    .withColumn("cancel_rate_pct",
        F.when(F.col("orders_placed") > 0, F.col("items_cancelled") / F.col("orders_placed") * 100).otherwise(0))

df_daily.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{SCHEMA}.clean_daily_metrics")

dm_count = spark.table(f"{CATALOG}.{SCHEMA}.clean_daily_metrics").count()
print(f"✅ clean_daily_metrics written: {dm_count:,} rows")

print(f"\n✅ clean_daily_metrics done. Building clean_sessions next...")

✅ clean_daily_metrics written: 65,680 rows

✅ clean_daily_metrics done. Building clean_sessions next...


In [0]:
# =============================================================================
# BUILD clean_sessions from session_base_v5 + identity_bridge_v5
# =============================================================================

df_session_raw = spark.table(f"{CATALOG}.{SCHEMA}.session_base_v5")
df_identity = spark.table(f"{CATALOG}.{SCHEMA}.identity_bridge_v5")

# Join identity bridge for user resolution
df_sessions_clean = df_session_raw.join(
    df_identity.select(
        F.col("stable_id").alias("ib_stable_id"),
        F.col("email").alias("resolved_email")
    ),
    df_session_raw["stable_id"] == F.col("ib_stable_id"),
    "left"
).drop("ib_stable_id")

# Add derived columns
df_sessions_clean = df_sessions_clean \
    .withColumn("is_identified", F.col("resolved_email").isNotNull()) \
    .withColumn("session_date", F.to_date("event_date"))

# Write
df_sessions_clean.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{SCHEMA}.clean_sessions")

cs_count = spark.table(f"{CATALOG}.{SCHEMA}.clean_sessions").count()
print(f"✅ clean_sessions written: {cs_count:,} rows")

# Keep reference for feature engineering
df_sessions = spark.table(f"{CATALOG}.{SCHEMA}.clean_sessions")

✅ clean_sessions written: 81,753,690 rows


In [0]:
# =============================================================================
# BUILD clean_product_events from product_events + identity_bridge_v5
# =============================================================================
# Filters to 16 key analytical event types, projects 16 of 49 columns

KEY_EVENTS = [
    "Product Viewed", "Product Seen", "Product Clicked",
    "Added To Bag", "Add to Bag Clicked", "Added to Bag",
    "Added to Wishlist", "Removed from Wishlist",
    "Size Selected",
    "Checkout Initiated", "Checkout Completed", "Checkout Clicked",
    "Bag Viewed", "Collection Viewed",
    "Search Submitted", "Search Query Action"
]

SELECT_COLS = [
    "user_id", "stable_id", "timestamp", "event_date", "event_name",
    "session_id", "app_vs_web",
    "product_id", "product_title", "product_type", "amount",
    "colour", "vendor", "collection", "collection_title", "size"
]

print(f"Filtering product_events to {len(KEY_EVENTS)} key event types...")
print(f"Projecting {len(SELECT_COLS)} of 49 columns (reduces I/O ~3x)")
print("(This processes 980M+ source rows — may take 10-15 minutes)")

df_product_filtered = spark.table(f"{CATALOG}.{SCHEMA}.product_events").select(
    *SELECT_COLS
).filter(
    F.col("event_name").isin(KEY_EVENTS)
)

# Join identity bridge (DEDUPLICATED to prevent fan-out)
df_identity_pe = spark.table(f"{CATALOG}.{SCHEMA}.identity_bridge_v5")
df_clean_product = df_product_filtered.join(
    df_identity_pe.select(
        F.col("stable_id").alias("ib_stable_id"),
        F.col("email").alias("resolved_email")
    ).dropDuplicates(["ib_stable_id"]),
    df_product_filtered["stable_id"] == F.col("ib_stable_id"),
    "left"
).drop("ib_stable_id")

# Add derived columns
df_clean_product = df_clean_product \
    .withColumn("is_identified", F.col("resolved_email").isNotNull()) \
    .withColumn("event_category",
        F.when(F.col("event_name").isin("Product Viewed", "Product Seen", "Product Clicked"), "Browse")
        .when(F.col("event_name").isin("Added To Bag", "Add to Bag Clicked", "Added to Bag"), "Add to Cart")
        .when(F.col("event_name").isin("Added to Wishlist", "Removed from Wishlist"), "Wishlist")
        .when(F.col("event_name").isin("Checkout Initiated", "Checkout Completed", "Checkout Clicked"), "Checkout")
        .when(F.col("event_name") == "Size Selected", "Size Selection")
        .when(F.col("event_name") == "Bag Viewed", "Cart View")
        .when(F.col("event_name") == "Collection Viewed", "Collection Browse")
        .when(F.col("event_name").isin("Search Submitted", "Search Query Action"), "Search")
        .otherwise("Other")
    )

# Write
df_clean_product.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{SCHEMA}.clean_product_events")

pe_count = spark.table(f"{CATALOG}.{SCHEMA}.clean_product_events").count()
print(f"\n✅ clean_product_events written: {pe_count:,} rows")
print(f"\n✅ Section 1 COMPLETE: All 4 clean tables built")

Filtering product_events to 16 key event types...
Projecting 16 of 49 columns (reduces I/O ~3x)
(This processes 980M+ source rows — may take 10-15 minutes)

✅ clean_product_events written: 575,699,582 rows

✅ Section 1 COMPLETE: All 4 clean tables built


## Section 2: Revenue Forecasting Model v2
Enhanced XGBoost + Prophet + Ridge ensemble with:
1. Rolling 180-day training window + anomaly exclusion
2. Session/UTM demand signals + lagged operational metrics
3. Regional sub-models (top 5 regions)
4. Weighted ensemble meta-learner
5. Conformal prediction intervals (MAPIE)
6. Model monitoring (PSI + drift)

In [0]:
# =============================================================================
# FEATURE ENGINEERING: Time series + Session signals + Operational metrics
# =============================================================================

# --- Load tables (safe if upstream cells skipped) ---
df_sessions = spark.table(f"{CATALOG}.{SCHEMA}.clean_sessions")

# --- National daily time series from clean_orders ---
df_placed = spark.table(f"{CATALOG}.{SCHEMA}.clean_orders").filter(F.col("event_name") == "Order Placed")

df_ts_national = df_placed.groupBy(F.col("order_date").alias("ds")).agg(
    F.sum("amount").alias("gross_revenue"),
    F.count("*").alias("order_count"),
    F.avg("amount").alias("avg_order_value"),
    F.sum("total_discounts").alias("total_discounts"),
    F.sum(F.when(F.col("is_discounted") == True, 1).otherwise(0)).alias("discounted_orders")
).orderBy("ds")

ts_pdf = df_ts_national.toPandas()
ts_pdf['ds'] = pd.to_datetime(ts_pdf['ds'])
# CRITICAL: Remove epoch dates (1970-01-01) from NULL order_date values
ts_pdf = ts_pdf[ts_pdf['ds'] >= '2024-01-01'].reset_index(drop=True)
ts_pdf = ts_pdf.sort_values('ds').reset_index(drop=True)
ts_pdf['net_revenue'] = ts_pdf['gross_revenue'] - ts_pdf['total_discounts']
ts_pdf['discount_pct'] = (ts_pdf['total_discounts'] / ts_pdf['gross_revenue'] * 100).fillna(0)
ts_pdf['discount_penetration'] = (ts_pdf['discounted_orders'] / ts_pdf['order_count'] * 100).fillna(0)

# --- Session features (daily) — optimized: date filter + approx_count_distinct ---
df_session_daily = df_sessions.filter(F.col("event_date") >= "2024-01-01").groupBy(F.col("event_date").alias("ds")).agg(
    F.count("*").alias("daily_session_count"),
    F.approx_count_distinct("stable_id").alias("unique_visitors"),
    F.avg(F.when(F.col("platform") == "App", 1).otherwise(0)).alias("app_ratio"),
    F.approx_count_distinct("utm_source").alias("utm_source_diversity"),
    F.avg(F.when(F.col("utm_medium") == "paid", 1).otherwise(0)).alias("paid_traffic_ratio")
).orderBy("ds")
session_pdf = df_session_daily.toPandas()
session_pdf['ds'] = pd.to_datetime(session_pdf['ds'])

# --- Operational metrics (daily aggregate) ---
df_dm = spark.table(f"{CATALOG}.{SCHEMA}.clean_daily_metrics")
metrics_pdf = df_dm.groupBy("order_date").agg(
    F.avg("return_rate_pct").alias("dm_return_rate"),
    F.avg("cancel_rate_pct").alias("dm_cancel_rate"),
    F.avg("discount_penetration_pct").alias("dm_discount_pen")
).orderBy("order_date").toPandas()
metrics_pdf.rename(columns={'order_date': 'ds'}, inplace=True)
metrics_pdf['ds'] = pd.to_datetime(metrics_pdf['ds'])

# --- Merge into master ---
ts_master = ts_pdf.merge(session_pdf, on='ds', how='left')
ts_master = ts_master.merge(metrics_pdf, on='ds', how='left')
ts_master.fillna(method='bfill', inplace=True)
ts_master.fillna(0, inplace=True)

# --- Create features ---
def create_features(df, target='gross_revenue'):
    df = df.copy()
    for lag in [1, 2, 3, 7, 14, 30]:
        df[f'lag_{lag}'] = df[target].shift(lag)
    df['diff_1'] = df[target].diff(1)
    df['diff_7'] = df[target].diff(7)
    for w in [7, 14, 30]:
        df[f'rolling_mean_{w}'] = df[target].rolling(w).mean()
        df[f'rolling_std_{w}'] = df[target].rolling(w).std()
    df['rolling_min_7'] = df[target].rolling(7).min()
    df['rolling_max_7'] = df[target].rolling(7).max()
    df['day_of_week'] = df['ds'].dt.dayofweek
    df['day_of_month'] = df['ds'].dt.day
    df['month'] = df['ds'].dt.month
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
    df['is_month_start'] = (df['day_of_month'] <= 3).astype(int)
    df['is_month_end'] = (df['day_of_month'] >= 28).astype(int)
    df['week_of_year'] = df['ds'].dt.isocalendar().week.astype(int)
    # Session signals (lagged 1 day)
    df['session_count_lag1'] = df['daily_session_count'].shift(1)
    df['session_7d_avg'] = df['daily_session_count'].rolling(7).mean().shift(1)
    df['app_ratio_lag1'] = df['app_ratio'].shift(1)
    df['utm_diversity_lag1'] = df['utm_source_diversity'].shift(1)
    df['paid_traffic_lag1'] = df['paid_traffic_ratio'].shift(1)
    df['visitors_lag1'] = df['unique_visitors'].shift(1)
    # Operational metrics (lagged)
    df['return_rate_lag7'] = df['dm_return_rate'].rolling(7).mean().shift(1)
    df['cancel_rate_lag7'] = df['dm_cancel_rate'].rolling(7).mean().shift(1)
    df['discount_pen_lag7'] = df['dm_discount_pen'].rolling(7).mean().shift(1)
    # Interaction
    df['rev_per_session'] = df['lag_1'] / df['session_count_lag1'].replace(0, np.nan)
    df['rev_per_session'].fillna(df['rev_per_session'].median(), inplace=True)
    return df.dropna().reset_index(drop=True)

ts_master['ds'] = pd.to_datetime(ts_master['ds'])
ts_featured = create_features(ts_master)

EXCLUDE = ['ds', 'gross_revenue', 'net_revenue', 'total_discounts', 'order_count',
           'avg_order_value', 'discounted_orders', 'discount_pct', 'discount_penetration',
           'daily_session_count', 'unique_visitors', 'app_ratio',
           'utm_source_diversity', 'paid_traffic_ratio',
           'dm_return_rate', 'dm_cancel_rate', 'dm_discount_pen', 'is_anomaly']
FEATURE_COLS = [c for c in ts_featured.columns if c not in EXCLUDE]

# --- Regional time series ---
ts_regional = df_placed.groupBy(
    F.col("order_date").alias("ds"), F.col("shipping_state_clean").alias("region")
).agg(
    F.sum("amount").alias("gross_revenue"),
    F.count("*").alias("order_count")
).orderBy("ds").toPandas()
ts_regional['ds'] = pd.to_datetime(ts_regional['ds'])

print(f"✅ Features: {len(FEATURE_COLS)} (was 28 in v1)")
print(f"   Time series: {len(ts_featured)} days")
print(f"   Date range: {ts_featured['ds'].min().date()} to {ts_featured['ds'].max().date()}")

/home/spark-61ee57c2-9b60-470c-8996-70/.ipykernel/11244/command-8045182445332960-1922581553:52: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  ts_master.fillna(method='bfill', inplace=True)
/home/spark-61ee57c2-9b60-470c-8996-70/.ipykernel/11244/command-8045182445332960-1922581553:87: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['rev_per_session'].fillna(df['rev_per_session'].median(), inplace=True)


✅ Features: 33 (was 28 in v1)
   Time series: 690 days
   Date range: 2024-07-25 to 2026-06-17


In [0]:
# =============================================================================
# ROLLING WINDOW + ANOMALY EXCLUSION + XGBoost v2 + REGIONAL MODELS
# =============================================================================

# Load anomaly dates (if gold_anomalies exists from prior run)
try:
    anomaly_dates = [row.anomaly_date for row in 
                     spark.table(f"{CATALOG}.{SCHEMA}.gold_anomalies").select("anomaly_date").collect()]
    anomaly_timestamps = set(pd.to_datetime(d) for d in anomaly_dates)
    print(f"Loaded {len(anomaly_timestamps)} anomaly dates to exclude")
except:
    anomaly_timestamps = set()
    print("No prior anomalies (first run)")

# Mark anomalies
ts_featured['is_anomaly'] = ts_featured['ds'].isin(anomaly_timestamps)

# --- Split: rolling window ---
test_start = len(ts_featured) - TEST_DAYS
roll_start = max(0, test_start - ROLLING_WINDOW_DAYS)
train_clean = ts_featured.iloc[roll_start:test_start]
train_clean = train_clean[~train_clean['is_anomaly']].reset_index(drop=True)
test_df = ts_featured.iloc[test_start:].reset_index(drop=True)

X_train = train_clean[FEATURE_COLS].values
y_train = train_clean['gross_revenue'].values
X_test = test_df[FEATURE_COLS].values
y_test = test_df['gross_revenue'].values

print(f"Train: {len(train_clean)} days (180d window, {ts_featured['is_anomaly'].sum()} anomalies excluded)")
print(f"Test: {len(test_df)} days")

# --- XGBoost v2 (national) ---
xgb_params = {
    'objective': 'reg:squarederror', 'max_depth': 6, 'learning_rate': 0.03,
    'subsample': 0.8, 'colsample_bytree': 0.7, 'min_child_weight': 5,
    'reg_alpha': 0.5, 'reg_lambda': 2.0, 'gamma': 0.1, 'seed': RANDOM_STATE
}
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=FEATURE_COLS)
dtest = xgb.DMatrix(X_test, label=y_test, feature_names=FEATURE_COLS)

model_xgb = xgb.train(xgb_params, dtrain, num_boost_round=300,
                       evals=[(dtest, 'test')], verbose_eval=False, early_stopping_rounds=30)
xgb_preds = model_xgb.predict(dtest)
# Safe MAPE (avoids divide-by-zero if any y_test == 0)
mask = y_test > 0
xgb_mape = np.mean(np.abs((y_test[mask] - xgb_preds[mask]) / y_test[mask])) * 100 if mask.any() else 999.0
print(f"\n✅ XGBoost v2: MAPE={xgb_mape:.2f}%")

# --- Regional sub-models ---
REG_FEATURES = ['lag_1','lag_2','lag_3','lag_7','lag_14','diff_1','diff_7',
                'rolling_mean_7','rolling_mean_14','rolling_std_7','rolling_std_14',
                'day_of_week','is_weekend','month','week_of_year']

regional_models = {}
regional_preds_combined = np.zeros(TEST_DAYS)
regional_actual_combined = np.zeros(TEST_DAYS)

for region in TOP_REGIONS + ['Other']:
    if region == 'Other':
        rdata = ts_regional[~ts_regional['region'].isin(TOP_REGIONS)].groupby('ds').agg(
            {'gross_revenue':'sum','order_count':'sum'}).reset_index()
    else:
        rdata = ts_regional[ts_regional['region'] == region].copy()
    rdata = rdata.sort_values('ds').reset_index(drop=True)
    if len(rdata) < 60:
        continue
    # Features
    for lag in [1,2,3,7,14]:
        rdata[f'lag_{lag}'] = rdata['gross_revenue'].shift(lag)
    rdata['diff_1'] = rdata['gross_revenue'].diff(1)
    rdata['diff_7'] = rdata['gross_revenue'].diff(7)
    for w in [7,14]:
        rdata[f'rolling_mean_{w}'] = rdata['gross_revenue'].rolling(w).mean()
        rdata[f'rolling_std_{w}'] = rdata['gross_revenue'].rolling(w).std()
    rdata['day_of_week'] = rdata['ds'].dt.dayofweek
    rdata['is_weekend'] = (rdata['day_of_week']>=5).astype(int)
    rdata['month'] = rdata['ds'].dt.month
    rdata['week_of_year'] = rdata['ds'].dt.isocalendar().week.astype(int)
    rdata = rdata.dropna().reset_index(drop=True)
    
    rt_start = max(0, len(rdata)-TEST_DAYS-ROLLING_WINDOW_DAYS)
    r_train = rdata.iloc[rt_start:len(rdata)-TEST_DAYS]
    r_test = rdata.iloc[len(rdata)-TEST_DAYS:]
    r_train = r_train[~r_train['ds'].isin(anomaly_timestamps)].reset_index(drop=True)
    if len(r_train)<30 or len(r_test)<5:
        continue
    
    rd = xgb.DMatrix(r_train[REG_FEATURES].values, label=r_train['gross_revenue'].values, feature_names=REG_FEATURES)
    rdt = xgb.DMatrix(r_test[REG_FEATURES].values, label=r_test['gross_revenue'].values, feature_names=REG_FEATURES)
    rm = xgb.train(xgb_params, rd, num_boost_round=200, evals=[(rdt,'t')], verbose_eval=False, early_stopping_rounds=20)
    rpreds = rm.predict(rdt)
    regional_models[region] = rm
    n = min(len(rpreds), TEST_DAYS)
    regional_preds_combined[:n] += rpreds[:n]
    regional_actual_combined[:n] += r_test['gross_revenue'].values[:n]

rmask = regional_actual_combined > 0
regional_mape = np.mean(np.abs((regional_actual_combined[rmask] - regional_preds_combined[rmask]) / regional_actual_combined[rmask])) * 100 if rmask.any() else 999.0
print(f"✅ Regional ({len(regional_models)} models): MAPE={regional_mape:.2f}%")

Loaded 16 anomaly dates to exclude
Train: 171 days (180d window, 16 anomalies excluded)
Test: 30 days

✅ XGBoost v2: MAPE=12.42%
✅ Regional (6 models): MAPE=7.65%


In [0]:
# =============================================================================
# PROPHET + RIDGE + ENSEMBLE
# =============================================================================

# --- Prophet ---
prophet_df = train_clean[['ds', 'gross_revenue']].copy()
prophet_df.columns = ['ds', 'y']

model_prophet = Prophet(
    yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False,
    changepoint_prior_scale=0.05, seasonality_prior_scale=10, holidays_prior_scale=10
)
model_prophet.add_country_holidays(country_name='IN')
model_prophet.add_seasonality(name='monthly', period=30.5, fourier_order=5)
model_prophet.fit(prophet_df)

future = model_prophet.make_future_dataframe(periods=TEST_DAYS)
forecast = model_prophet.predict(future)
prophet_preds = forecast.tail(TEST_DAYS)['yhat'].values
prophet_mape = np.mean(np.abs((y_test[mask] - prophet_preds[mask]) / y_test[mask])) * 100 if mask.any() else 999.0
print(f"✅ Prophet: MAPE={prophet_mape:.2f}%")

# --- Ridge ---
scaler = StandardScaler()
X_tr_s = np.nan_to_num(scaler.fit_transform(X_train))
X_te_s = np.nan_to_num(scaler.transform(X_test))
model_ridge = Ridge(alpha=100.0)
model_ridge.fit(X_tr_s, y_train)
ridge_preds = model_ridge.predict(X_te_s)
ridge_mape = np.mean(np.abs((y_test[mask] - ridge_preds[mask]) / y_test[mask])) * 100 if mask.any() else 999.0
print(f"✅ Ridge: MAPE={ridge_mape:.2f}%")

# --- Ensemble (inverse-MAPE weighted) ---
models_info = {'xgb': (xgb_preds, xgb_mape), 'prophet': (prophet_preds, prophet_mape), 'ridge': (ridge_preds, ridge_mape)}
total_inv = sum(1.0/m for _, m in models_info.values())
weights = {k: (1.0/m)/total_inv for k, (_, m) in models_info.items()}

ensemble_preds = sum(w * models_info[k][0] for k, w in weights.items())
ensemble_mape = np.mean(np.abs((y_test[mask] - ensemble_preds[mask]) / y_test[mask])) * 100 if mask.any() else 999.0

# Blend with regional
blended_preds = 0.6 * ensemble_preds + 0.4 * regional_preds_combined
blended_mape = np.mean(np.abs((y_test[mask] - blended_preds[mask]) / y_test[mask])) * 100 if mask.any() else 999.0

# Pick best
if blended_mape < ensemble_mape:
    final_preds, final_mape = blended_preds, blended_mape
    final_method = "Blended (60% ensemble + 40% regional)"
else:
    final_preds, final_mape = ensemble_preds, ensemble_mape
    final_method = "Weighted Ensemble"

final_preds = np.nan_to_num(final_preds, nan=np.nanmean(y_test))
final_mae = mean_absolute_error(y_test, final_preds)
final_rmse = np.sqrt(mean_squared_error(y_test, final_preds))

print(f"\n⭐ BEST: {final_method}")
print(f"   MAPE={final_mape:.2f}% | MAE=₹{final_mae:,.0f} | RMSE=₹{final_rmse:,.0f}")
print(f"   Weights: {', '.join(f'{k}={v:.2f}' for k,v in weights.items())}")

/local_disk0/.ephemeral_nfs/envs/pythonEnv-61ee57c2-9b60-470c-8996-70f8ac8c5455/lib/python3.12/site-packages/prophet/make_holidays.py:49: UserWarning: Requested Holidays are available only from 2001 to 2035.
  return set(country_holidays(language="en_US", years=np.arange(1995, 2045)).values())
18:24:48 - cmdstanpy - INFO - Chain [1] start processing
18:24:48 - cmdstanpy - INFO - Chain [1] done processing


✅ Prophet: MAPE=52.65%
✅ Ridge: MAPE=12.91%

⭐ BEST: Blended (60% ensemble + 40% regional)
   MAPE=9.80% | MAE=₹93,773 | RMSE=₹160,044
   Weights: xgb=0.45, prophet=0.11, ridge=0.44


In [0]:
# =============================================================================
# CONFORMAL PREDICTION (MAPIE) + MODEL MONITORING
# =============================================================================
from sklearn.base import BaseEstimator, RegressorMixin
from mapie.regression import CrossConformalRegressor

class XGBWrapper(BaseEstimator, RegressorMixin):
    def __init__(self): self.model_ = None
    def fit(self, X, y):
        d = xgb.DMatrix(X, label=y, feature_names=FEATURE_COLS)
        self.model_ = xgb.train(xgb_params, d, num_boost_round=300, verbose_eval=False)
        return self
    def predict(self, X):
        return self.model_.predict(xgb.DMatrix(X, feature_names=FEATURE_COLS))

mapie = CrossConformalRegressor(estimator=XGBWrapper(), confidence_level=[0.80, 0.95], method="plus", cv=5)
mapie.fit_conformalize(X_train, y_train)
y_pred_mapie, y_pis = mapie.predict_interval(X_test)

ci_lower_80, ci_upper_80 = y_pis[:, 0, 0], y_pis[:, 1, 0]
ci_lower_95, ci_upper_95 = y_pis[:, 0, 1], y_pis[:, 1, 1]

coverage_80 = np.mean((y_test >= ci_lower_80) & (y_test <= ci_upper_80)) * 100
coverage_95 = np.mean((y_test >= ci_lower_95) & (y_test <= ci_upper_95)) * 100

print(f"✅ Conformal Prediction (MAPIE):")
print(f"   Coverage@80%: {coverage_80:.1f}% | Coverage@95%: {coverage_95:.1f}%")

# --- Model Monitoring ---
def calc_psi(expected, actual, n_bins=10):
    bp = np.percentile(expected, np.linspace(0, 100, n_bins+1))
    bp[0], bp[-1] = -np.inf, np.inf
    e_pct = (np.histogram(expected, bp)[0] + 1) / (len(expected) + n_bins)
    a_pct = (np.histogram(actual, bp)[0] + 1) / (len(actual) + n_bins)
    return float(np.sum((a_pct - e_pct) * np.log(a_pct / e_pct)))

train_preds = model_xgb.predict(dtrain)
psi = calc_psi(train_preds, xgb_preds)

# Feature drift
n_drifted = sum(1 for i in range(X_train.shape[1])
                if ks_2samp(X_train[:, i], X_test[:, i]).pvalue < 0.05)

# Confidence score
# Safe confidence (handles NaN or inf MAPE)
global_confidence = min(0.95, max(0.5, 1.0 - (final_mape if np.isfinite(final_mape) else 30.0) / 100))

print(f"\n✅ Model Monitoring:")
print(f"   PSI: {psi:.4f} | Drifted features: {n_drifted}/{len(FEATURE_COLS)}")
print(f"   Global confidence: {global_confidence:.3f}")
print(f"\n✅ Section 2 complete: Model v2 trained")

✅ Conformal Prediction (MAPIE):
   Coverage@80%: 66.7% | Coverage@95%: 80.0%

✅ Model Monitoring:
   PSI: 1.0310 | Drifted features: 21/33
   Global confidence: 0.902

✅ Section 2 complete: Model v2 trained


In [0]:
# =============================================================================
# SHAP FEATURE IMPORTANCE (using XGBoost native pred_contribs)
# No shap library needed — uses get_booster().predict(pred_contribs=True)
# =============================================================================

# Get SHAP values for test set
shap_values = model_xgb.predict(dtest, pred_contribs=True)
# Last column is bias term, exclude it
shap_features = shap_values[:, :-1]

# Global feature importance (mean absolute SHAP)
shap_importance = pd.DataFrame({
    'feature': FEATURE_COLS,
    'mean_abs_shap': np.abs(shap_features).mean(axis=0)
}).sort_values('mean_abs_shap', ascending=False)

print("✅ SHAP Feature Importance (Top 10):")
print(f"{'Feature':<30} {'Mean |SHAP|':<15} {'% Contribution'}")
print("-" * 60)
total_shap = shap_importance['mean_abs_shap'].sum()
for _, row in shap_importance.head(10).iterrows():
    pct = row['mean_abs_shap'] / total_shap * 100
    print(f"{row['feature']:<30} {row['mean_abs_shap']:<15,.0f} {pct:.1f}%")

# Per-prediction SHAP (top 5 features per test day) — stored for recommendations
def get_top_shap(idx, n=5):
    """Get top N SHAP contributors for a single prediction."""
    vals = shap_features[idx]
    top_idx = np.argsort(np.abs(vals))[::-1][:n]
    return {FEATURE_COLS[i]: round(float(vals[i]), 2) for i in top_idx}

# Store SHAP for each test day (used by gold_region_forecasts and recommendations)
shap_per_day = [get_top_shap(i) for i in range(len(shap_features))]

# Global top 5 for recommendation summaries
SHAP_TOP5_GLOBAL = shap_importance.head(5)['feature'].tolist()
SHAP_GLOBAL_DICT = {row['feature']: round(float(row['mean_abs_shap']), 2) 
                    for _, row in shap_importance.head(10).iterrows()}

print(f"\n✅ SHAP computed: {len(shap_features)} predictions × {len(FEATURE_COLS)} features")
print(f"   Top driver: {SHAP_TOP5_GLOBAL[0]} ({shap_importance.iloc[0]['mean_abs_shap']:,.0f})")

✅ SHAP Feature Importance (Top 10):
Feature                        Mean |SHAP|     % Contribution
------------------------------------------------------------
rolling_min_7                  65,512          19.5%
lag_1                          46,590          13.9%
session_7d_avg                 44,374          13.2%
diff_1                         37,802          11.2%
rolling_std_7                  29,579          8.8%
diff_7                         26,925          8.0%
rolling_max_7                  18,152          5.4%
rolling_mean_7                 17,573          5.2%
rolling_std_14                 12,632          3.8%
rolling_mean_14                4,692           1.4%

✅ SHAP computed: 30 predictions × 33 features
   Top driver: rolling_min_7 (65,512)


## Section 3: Decision Intelligence Engine
Anomalies, causal inference, scenario simulation, budget optimization, traceable recommendations.

In [0]:
# =============================================================================
# ANOMALY DETECTION + CAUSAL INFERENCE
# =============================================================================

# --- Anomaly Detection: IsolationForest + 3-sigma ---
rev_series = ts_featured[['ds', 'gross_revenue']].copy()
rev_series['z_score'] = stats.zscore(rev_series['gross_revenue'])
rev_series['rolling_mean'] = rev_series['gross_revenue'].rolling(30).mean()
rev_series['rolling_std'] = rev_series['gross_revenue'].rolling(30).std()
rev_series['expected_value'] = rev_series['rolling_mean']
rev_series['deviation_pct'] = ((rev_series['gross_revenue'] - rev_series['expected_value']) / rev_series['expected_value'] * 100).fillna(0)

# IsolationForest
iso = IsolationForest(contamination=0.05, random_state=RANDOM_STATE)
rev_series['iso_anomaly'] = iso.fit_predict(rev_series[['gross_revenue']].fillna(0))

# Combine: statistical + isolation forest
rev_series['is_anomaly'] = (
    (rev_series['iso_anomaly'] == -1) | (rev_series['z_score'].abs() > 2.5)
)

def get_severity(z):
    az = abs(z)
    if az > 5: return 'CRITICAL'
    elif az > 3: return 'WARNING'
    elif az > 2: return 'INFO'
    return None

rev_series['severity'] = rev_series['z_score'].apply(get_severity)
anomalies_df = rev_series[rev_series['is_anomaly'] & rev_series['severity'].notna()].copy()
anomalies_df['revenue_impact'] = anomalies_df['gross_revenue'] - anomalies_df['expected_value']

print(f"✅ Anomaly Detection: {len(anomalies_df)} anomalies")
print(f"   CRITICAL: {(anomalies_df['severity']=='CRITICAL').sum()}")
print(f"   WARNING: {(anomalies_df['severity']=='WARNING').sum()}")
print(f"   INFO: {(anomalies_df['severity']=='INFO').sum()}")

# --- Causal Inference (DoWhy) ---
import dowhy
from dowhy import CausalModel

# Regional profitability for causal analysis
df_region = df_placed.groupBy("shipping_state_clean").agg(
    F.count("*").alias("orders"),
    F.sum("amount").alias("gross_revenue"),
    F.sum("total_discounts").alias("total_discounts"),
    F.avg("amount").alias("aov"),
    F.avg(F.when(F.col("is_discounted")==True, 1).otherwise(0)).alias("discount_rate")
).toPandas()
df_region.columns = ['region','orders','gross_revenue','total_discounts','aov','discount_rate']
df_region['net_revenue'] = df_region['gross_revenue'] - df_region['total_discounts']
df_region['retention_pct'] = (df_region['net_revenue'] / df_region['gross_revenue'] * 100)
df_region = df_region.sort_values('gross_revenue', ascending=False).head(15)

causal_df = df_region[['region','orders','gross_revenue','aov','discount_rate','retention_pct']].copy()
causal_df['log_orders'] = np.log1p(causal_df['orders'])

try:
    cm = CausalModel(data=causal_df, treatment='discount_rate', outcome='retention_pct',
                     common_causes=['log_orders', 'aov'])
    identified = cm.identify_effect(proceed_when_unidentifiable=True)
    estimate = cm.estimate_effect(identified, method_name="backdoor.linear_regression")
    causal_effect = estimate.value
    refutation = cm.refute_estimate(identified, estimate, method_name="placebo_treatment_refuter",
                                    placebo_type="permute", num_simulations=50)
    CAUSAL_VALIDATED = abs(refutation.new_effect) < abs(causal_effect) * 0.5
    print(f"\n✅ Causal Inference: effect={causal_effect:.4f}, validated={CAUSAL_VALIDATED}")
except Exception as e:
    causal_effect = -1.49
    CAUSAL_VALIDATED = False
    print(f"\u26a0️ Causal fallback: {e}")

✅ Anomaly Detection: 16 anomalies
   CRITICAL: 8
   INFO: 2


/databricks/python/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:430: UserWarning: `kurtosistest` p-value may be inaccurate with fewer than 20 observations; only n=15 observations were given.
  return hypotest_fun_in(*args, **kwds)
/databricks/python/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:430: UserWarning: `kurtosistest` p-value may be inaccurate with fewer than 20 observations; only n=15 observations were given.
  return hypotest_fun_in(*args, **kwds)
/databricks/python/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:430: UserWarning: `kurtosistest` p-value may be inaccurate with fewer than 20 observations; only n=15 observations were given.
  return hypotest_fun_in(*args, **kwds)
/databricks/python/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:430: UserWarning: `kurtosistest` p-value may be inaccurate with fewer than 20 observations; only n=15 observations were given.
  return hypotest_fun_in(*args, **kwds)
/databricks/


✅ Causal Inference: effect=-35.1032, validated=True


In [0]:
# =============================================================================
# SCENARIO SIMULATION (Monte Carlo) + OR-TOOLS BUDGET OPTIMIZATION
# =============================================================================

# --- Monte Carlo Scenarios ---
base_annual = ts_pdf['gross_revenue'].sum() / (len(ts_pdf) / 365)

scenarios = [
    {'name': 'Conservative Discount (-3pp)', 'discount_change_pp': -3, 'return_change_pp': 0, 'cancel_change_pp': 0, 'cac_change_pct': 0},
    {'name': 'Aggressive Discount (-5pp)', 'discount_change_pp': -5, 'return_change_pp': 0, 'cancel_change_pp': 0, 'cac_change_pct': 0},
    {'name': 'Return Reduction (-5pp)', 'discount_change_pp': 0, 'return_change_pp': -5, 'cancel_change_pp': 0, 'cac_change_pct': 0},
    {'name': 'Cancel Reduction (-5pp)', 'discount_change_pp': 0, 'return_change_pp': 0, 'cancel_change_pp': -5, 'cac_change_pct': 0},
    {'name': 'Combined (disc-3 + return-3)', 'discount_change_pp': -3, 'return_change_pp': -3, 'cancel_change_pp': 0, 'cac_change_pct': 0},
    {'name': 'CAC Reduction (-20%)', 'discount_change_pp': 0, 'return_change_pp': 0, 'cancel_change_pp': 0, 'cac_change_pct': -20},
]

scenario_results = []
for s in scenarios:
    sims = []
    for _ in range(1000):
        impact = (
            causal_effect * s['discount_change_pp'] * base_annual / 100 +
            s['return_change_pp'] * base_annual * 0.005 +
            s['cancel_change_pp'] * base_annual * 0.004 +
            s['cac_change_pct'] * base_annual * 0.002 +
            np.random.normal(0, base_annual * 0.02)
        )
        sims.append(impact)
    sims = np.array(sims)
    scenario_results.append({
        'scenario': s['name'],
        'revenue_impact_mean': float(np.mean(sims)),
        'revenue_impact_median': float(np.median(sims)),
        'ci_lower_95': float(np.percentile(sims, 2.5)),
        'ci_upper_95': float(np.percentile(sims, 97.5)),
        'ci_lower_80': float(np.percentile(sims, 10)),
        'ci_upper_80': float(np.percentile(sims, 90)),
        'probability_positive': float(np.mean(sims > 0)),
        'max_downside': float(np.min(sims)),
        'max_upside': float(np.max(sims)),
        'std': float(np.std(sims)),
        'n_simulations': 1000
    })

print(f"✅ {len(scenario_results)} scenarios simulated (1000 iterations each)")

# --- OR-Tools Budget Optimization ---
from ortools.linear_solver import pywraplp

regions_list = df_region['region'].tolist()
retention_list = df_region['retention_pct'].tolist()
revenue_list = df_region['gross_revenue'].tolist()

solver = pywraplp.Solver.CreateSolver('SCIP')
n = len(regions_list)
TOTAL_BUDGET = 50_000_000
allocs = [solver.NumVar(1_000_000, 25_000_000, f'a_{i}') for i in range(n)]
solver.Add(sum(allocs) <= TOTAL_BUDGET)

objective = solver.Objective()
for i in range(n):
    coeff = (retention_list[i]/100) * 1.5 * (revenue_list[i] / max(revenue_list))
    objective.SetCoefficient(allocs[i], coeff)
objective.SetMaximization()
solver.Solve()

optimal_alloc = []
for i in range(n):
    a = allocs[i].solution_value()
    custs = a / 500
    exp_rev = custs * (revenue_list[i]/max(revenue_list)) * 2000 * 1.5 * (retention_list[i]/100)
    optimal_alloc.append({
        'region': regions_list[i], 'allocation': a,
        'allocation_pct': a/TOTAL_BUDGET*100,
        'expected_revenue': exp_rev,
        'roi_multiplier': exp_rev/a if a > 0 else 0,
        'retention_pct': retention_list[i]
    })
optimal_alloc = pd.DataFrame(optimal_alloc).sort_values('allocation', ascending=False)

print(f"✅ Budget optimization: ₹5Cr allocated across {n} regions")
print(f"   Top allocation: {optimal_alloc.iloc[0]['region']} ({optimal_alloc.iloc[0]['allocation_pct']:.1f}%)")

✅ 6 scenarios simulated (1000 iterations each)
✅ Budget optimization: ₹5Cr allocated across 15 regions
   Top allocation: Maharashtra (50.0%)


In [0]:
# =============================================================================
# RECOMMENDATION ENGINE WITH FULL TRACEABILITY
# =============================================================================

def calculate_confidence(n_obs, mape_val, feature_stability):
    vol_score = min(1.0, n_obs / 500)
    accuracy_score = max(0, 1.0 - mape_val / 50)
    return round(0.4 * vol_score + 0.4 * accuracy_score + 0.2 * feature_stability, 3)

recommendations = []

# From scenarios
for s in scenario_results:
    rec_id = str(uuid.uuid4())
    action = s['scenario']
    impact = s['revenue_impact_mean']
    direction = 'positive' if impact > 0 else 'negative'
    conf = calculate_confidence(len(train_clean), final_mape, 0.85)
    recommendations.append({
        'recommendation_id': rec_id,
        'timestamp': datetime.now().isoformat(),
        'question': f"What is the impact of {action.lower()}?",
        'region': 'All Regions',
        'recommended_action': action,
        'predicted_impact_annual': impact,
        'ci_lower_95': s['ci_lower_95'],
        'ci_upper_95': s['ci_upper_95'],
        'confidence_score': conf,
        'model_version': MODEL_VERSION,
        'training_data_range': f"{ts_featured['ds'].min().date()} to {ts_featured['ds'].max().date()}",
        'features_used': json.dumps(FEATURE_COLS[:10]),
        'shap_contributions': json.dumps(SHAP_GLOBAL_DICT),
        'supporting_evidence': json.dumps({'probability_positive': s['probability_positive'], 'n_sims': 1000, 'top_drivers': SHAP_TOP5_GLOBAL}),
        'causal_validated': CAUSAL_VALIDATED,
        'status': 'PENDING',
        'executive_summary': f"RECOMMENDATION: {action}. Expected annual impact: \u20b9{impact:,.0f} ({direction}). 95% CI: [\u20b9{s['ci_lower_95']:,.0f}, \u20b9{s['ci_upper_95']:,.0f}]. Confidence: {conf:.0%}."
    })

# From budget optimization
for _, row in optimal_alloc.iterrows():
    rec_id = str(uuid.uuid4())
    conf = calculate_confidence(int(row.get('allocation',0)/1000), final_mape, 0.85)
    recommendations.append({
        'recommendation_id': rec_id,
        'timestamp': datetime.now().isoformat(),
        'question': f"How much budget for {row['region']}?",
        'region': row['region'],
        'recommended_action': f"Allocate \u20b9{row['allocation']:,.0f} ({row['allocation_pct']:.1f}%)",
        'predicted_impact_annual': row['expected_revenue'],
        'ci_lower_95': row['expected_revenue'] * 0.7,
        'ci_upper_95': row['expected_revenue'] * 1.4,
        'confidence_score': conf,
        'model_version': MODEL_VERSION,
        'training_data_range': f"{ts_featured['ds'].min().date()} to {ts_featured['ds'].max().date()}",
        'features_used': json.dumps(['retention', 'revenue_share', 'cac']),
        'shap_contributions': json.dumps(SHAP_GLOBAL_DICT),
        'supporting_evidence': json.dumps({'roi': row['roi_multiplier'], 'method': 'OR-Tools SCIP', 'top_drivers': SHAP_TOP5_GLOBAL}),
        'causal_validated': CAUSAL_VALIDATED,
        'status': 'PENDING',
        'executive_summary': f"RECOMMENDATION: Allocate \u20b9{row['allocation']:,.0f} to {row['region']} ({row['allocation_pct']:.1f}% of budget). Expected ROI: {row['roi_multiplier']:.2f}x."
    })

rec_df = pd.DataFrame(recommendations)
print(f"✅ {len(rec_df)} recommendations generated with UUID traceability")
print(f"\n✅ Section 3 complete: Decision Intelligence done")

✅ 21 recommendations generated with UUID traceability

✅ Section 3 complete: Decision Intelligence done


## Section 4: Write Gold Tables to Unity Catalog
All 7 gold tables written in overwrite mode. Genie space auto-refreshes.

In [0]:
# =============================================================================
# WRITE ALL 7 GOLD TABLES
# =============================================================================

# 1. gold_profitability_drivers
prof = df_region.copy()
prof['confidence_score'] = prof.apply(lambda r: calculate_confidence(int(r['orders']), final_mape, 0.85), axis=1)
prof['model_version'] = MODEL_VERSION
prof['updated_at'] = datetime.now().isoformat()

spark.createDataFrame(prof).write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.{SCHEMA}.gold_profitability_drivers")
print(f"✅ gold_profitability_drivers: {len(prof)} rows")

# 2. gold_region_forecasts (with conformal intervals)
forecast_rows = []
for i in range(len(test_df)):
    forecast_rows.append({
        'forecast_date': test_df.iloc[i]['ds'].isoformat(),
        'predicted_revenue': float(final_preds[i]),
        'ci_lower_80': float(ci_lower_80[i]),
        'ci_upper_80': float(ci_upper_80[i]),
        'ci_lower_95': float(ci_lower_95[i]),
        'ci_upper_95': float(ci_upper_95[i]),
        'actual_revenue': float(y_test[i]),
        'mape_pct': float(abs(y_test[i] - final_preds[i]) / y_test[i] * 100),
        'confidence_score': global_confidence,
        'model_version': MODEL_VERSION,
        'method': final_method
    })
spark.createDataFrame(pd.DataFrame(forecast_rows)).write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.{SCHEMA}.gold_region_forecasts")
print(f"✅ gold_region_forecasts: {len(forecast_rows)} rows")

# 3. gold_recommendations
spark.createDataFrame(rec_df).write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.{SCHEMA}.gold_recommendations")
print(f"✅ gold_recommendations: {len(rec_df)} rows")

# 4. gold_anomalies
anom_out = anomalies_df[['ds','gross_revenue','expected_value','deviation_pct','severity','revenue_impact','z_score']].copy()
anom_out['metric_name'] = 'gross_revenue'
anom_out['model_version'] = MODEL_VERSION
anom_out['detected_at'] = datetime.now().isoformat()
anom_out.columns = ['anomaly_date','current_value','expected_value','deviation_pct','severity','revenue_impact','z_score','metric_name','model_version','detected_at']
spark.createDataFrame(anom_out).write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.{SCHEMA}.gold_anomalies")
print(f"✅ gold_anomalies: {len(anom_out)} rows")

# 5. gold_scenarios
scen_df = pd.DataFrame(scenario_results)
scen_df['model_version'] = MODEL_VERSION
scen_df['run_timestamp'] = datetime.now().isoformat()
scen_df['confidence_score'] = global_confidence
spark.createDataFrame(scen_df).write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.{SCHEMA}.gold_scenarios")
print(f"✅ gold_scenarios: {len(scen_df)} rows")

# 6. gold_model_performance
perf = pd.DataFrame([{
    'timestamp': datetime.now().isoformat(),
    'model_version': MODEL_VERSION,
    'mae': float(final_mae), 'rmse': float(final_rmse), 'mape_pct': float(final_mape),
    'psi_predictions': float(psi), 'features_drifted': int(n_drifted),
    'prediction_coverage_pct': float(coverage_80),
    'retrain_triggered': psi > 0.2,
    'confidence_score': global_confidence,
    'training_rows': int(len(train_clean)), 'test_rows': int(len(test_df))
}])
spark.createDataFrame(perf).write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.{SCHEMA}.gold_model_performance")
print(f"✅ gold_model_performance: 1 row")

# 7. gold_decision_history (schema only)
history_schema = StructType([
    StructField('recommendation_id', StringType()),
    StructField('action_date', StringType()),
    StructField('predicted_outcome', DoubleType()),
    StructField('actual_outcome', DoubleType()),
    StructField('prediction_error_pct', DoubleType()),
    StructField('model_version', StringType()),
    StructField('was_accurate', BooleanType())
])
spark.createDataFrame([], history_schema).write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.{SCHEMA}.gold_decision_history")
print(f"✅ gold_decision_history: schema ready")

# --- MLflow Log ---
with mlflow.start_run(run_name=f"pipeline_{MODEL_VERSION}") as run:
    mlflow.log_params({'model_version': MODEL_VERSION, 'rolling_window': ROLLING_WINDOW_DAYS,
                       'n_features': len(FEATURE_COLS), 'method': final_method})
    mlflow.log_metrics({'mape': final_mape, 'mae': final_mae, 'rmse': final_rmse,
                        'coverage_80': coverage_80, 'coverage_95': coverage_95, 'psi': psi})
    print(f"\n✅ MLflow logged: {run.info.run_id}")

print(f"\n{'='*60}")
print(f"ALL 7 GOLD TABLES WRITTEN TO {CATALOG}.{SCHEMA}")
print(f"{'='*60}")

✅ gold_profitability_drivers: 15 rows
✅ gold_region_forecasts: 30 rows
✅ gold_recommendations: 21 rows
✅ gold_anomalies: 16 rows
✅ gold_scenarios: 6 rows
✅ gold_model_performance: 1 row
✅ gold_decision_history: schema ready

✅ MLflow logged: d3fe98307fe34d6daa38622311b8c7a2

ALL 7 GOLD TABLES WRITTEN TO ameyam_statsig_databricks_warehouse.default


In [0]:
# =============================================================================
# SECTION 5: FINAL VALIDATION
# =============================================================================

print("\n" + "="*70)
print("   VIRGIO DECISION INTELLIGENCE PIPELINE — SYSTEM STATUS")
print("="*70)

gold_tables = ['gold_profitability_drivers', 'gold_region_forecasts', 'gold_recommendations',
               'gold_anomalies', 'gold_scenarios', 'gold_model_performance', 'gold_decision_history']

print(f"\n{'Table':<40} {'Rows':<10} {'Status'}")
print("-"*65)
all_ok = True
for t in gold_tables:
    try:
        c = spark.table(f"{CATALOG}.{SCHEMA}.{t}").count()
        print(f"{t:<40} {c:<10} ✅")
    except Exception as e:
        print(f"{t:<40} {'N/A':<10} ❌ {str(e)[:30]}")
        all_ok = False

print(f"\n{'='*70}")
print(f"MODEL PERFORMANCE ({MODEL_VERSION}):")
print(f"  MAPE: {final_mape:.2f}% (target <14%, baseline was 19.85%)")
print(f"  MAE: ₹{final_mae:,.0f} | RMSE: ₹{final_rmse:,.0f}")
print(f"  Coverage@80%: {coverage_80:.1f}% | Coverage@95%: {coverage_95:.1f}%")
print(f"  Method: {final_method}")
print(f"  Features: {len(FEATURE_COLS)} | Regional models: {len(regional_models)}")
print(f"  PSI: {psi:.4f} | Drifted: {n_drifted}/{len(FEATURE_COLS)}")
print(f"  Confidence: {global_confidence:.3f}")
print(f"  Causal validated: {CAUSAL_VALIDATED}")
print(f"\nDECISION INTELLIGENCE:")
print(f"  Anomalies: {len(anomalies_df)} | Scenarios: {len(scenario_results)}")
print(f"  Recommendations: {len(rec_df)} | Budget optimized: ₹5Cr")
print(f"\nDATA FRESHNESS:")
co_count = spark.table(f"{CATALOG}.{SCHEMA}.clean_orders").count()
dm_count = spark.table(f"{CATALOG}.{SCHEMA}.clean_daily_metrics").count()
print(f"  clean_orders: {co_count:,} rows")
print(f"  clean_daily_metrics: {dm_count:,} rows")
print(f"  Date range: {ts_featured['ds'].min().date()} to {ts_featured['ds'].max().date()}")
print(f"\n{'='*70}")
status = "✅ PIPELINE SUCCESS" if all_ok else "❌ PIPELINE HAS ISSUES"
print(f"  {status} — Genie Space ready for queries")
print(f"  Completed at: {datetime.now()}")
print(f"{'='*70}")


   VIRGIO DECISION INTELLIGENCE PIPELINE — SYSTEM STATUS

Table                                    Rows       Status
-----------------------------------------------------------------
gold_profitability_drivers               15         ✅
gold_region_forecasts                    30         ✅
gold_recommendations                     21         ✅
gold_anomalies                           16         ✅
gold_scenarios                           6          ✅
gold_model_performance                   1          ✅
gold_decision_history                    0          ✅

MODEL PERFORMANCE (v3.0_consolidated_pipeline):
  MAPE: 9.80% (target <14%, baseline was 19.85%)
  MAE: ₹93,773 | RMSE: ₹160,044
  Coverage@80%: 66.7% | Coverage@95%: 80.0%
  Method: Blended (60% ensemble + 40% regional)
  Features: 33 | Regional models: 6
  PSI: 1.0310 | Drifted: 21/33
  Confidence: 0.902
  Causal validated: True

DECISION INTELLIGENCE:
  Anomalies: 16 | Scenarios: 6
  Recommendations: 21 | Budget optimized: ₹5Cr

D